In [ ]:
import numpy as np
from  numpy import deg2rad as d2r
from  numpy import array as arr

from space_traj_opt import OUT_DIR
from space_traj_opt.noc.noc_controller import Vehicle, NOCGuidance
from space_traj_opt.math.constants import EARTH_R

from space_traj_opt.math.integrator import integrate

from space_traj_opt.postprocessing.plotting import plot

STANDARD_GRAV = 9.80665

## Electron Rocket Parameters

In [ ]:
n_engines_s1 = 9
n_engines_s2 = 1
isp_s1 = 311.0
engine_thrust_s1 = n_engines_s1*24910.04  # N Average between sl and vac
isp_s2 = 343.0
engine_thrust_s2 = n_engines_s2* 25_000.0  # N
s1_vch_params = (engine_thrust_s1, isp_s1, True)
s2_vch_params = (engine_thrust_s2, isp_s2, False)

fairing_mass = 50.0
farinig_timing = 184.0 - 162.0 # sec
payload = 250.0
s1_dry_mass = 1076.47308279  
s2_dry_mass = 257.90093739  

s1_wet_mass = 10047.082106064723
s2_wet_mass = 2602.454913676189
total_mass = 12949.537019740912

mdot_s1 = engine_thrust_s1 / STANDARD_GRAV / isp_s1
mdot_s2 = engine_thrust_s2 / STANDARD_GRAV / isp_s2

## Initial Guesses

In [ ]:
mu_earth = 3.986004418e14
earth_r = 6_378_000.0 # m
circ_orbit_alt = 200_000.0 
v_circ = np.sqrt(mu_earth / (earth_r + circ_orbit_alt))

# Guesses 
s2_sep_mass = s2_wet_mass + payload + fairing_mass
x0 = arr([
    [0.0,   6462918.0 , 0.0, 2457.0, 997.0, 200.0, 2688.0]
])




In [ ]:
ctrl =NOCGuidance.from_gain_table(OUT_DIR /"noc" / "second_stage_ascent_gain_table.npz")


vehicle = Vehicle(
    ctrl,
    plant_params=s2_vch_params
)

In [ ]:
x0[0]

In [ ]:
ctrl.update(t= 0.0, x=x0[0] )

## Build The problem
Builds the decision vector and bounds 

In [ ]:

t = 291.12

t_sol, sol_y = integrate(
    vehicle.run, 
    t_span=[0.0, t], 
    t_eval= np.linspace(0.0, t,200),
    y0=x0[0],    
    #args=(vch_params,)
)

In [ ]:
from space_traj_opt.postprocessing.data_client import CSVClient

client = CSVClient("noc/second_stage_ascent_vz_plus")
#client = CSVClient("noc/second_stage_ascent_x_plus")

In [ ]:
# state = client.get_channels(["rx", "ry", "rz", "vx", "vy", "vz", "m"])
control = client.get_channels(["pitch", "yaw"])
a_mag = np.linalg.norm(client.get_channels(["ax", "ay", "az"]), axis=1)

time = client.get_time()


In [ ]:
a_mag.shape

In [ ]:
state = sol_y.T

pos = state[:, 0:3]
vel = state[:, 3:6]
mass = state[:,6]

In [ ]:

v_eci_mag = np.linalg.norm(vel, axis=1) 

In [ ]:
ctrl.table.gain.shape

In [ ]:
from space_traj_opt.math.orbital_calcs import rv_to_orbital_elements
from space_traj_opt.models.controller3d import flight_path_angle

fpa = flight_path_angle(pos, vel)


pitch_values = []
yaw_values = []
a, e, i ,_,_,_  = rv_to_orbital_elements(pos, vel)

for idx, (t, cur_x) in enumerate(zip(t_sol, state)):
    pitch, yaw = ctrl.table.guidance(-cur_x[6], cur_x[0:6] )
    pitch_values.append(pitch)
    yaw_values.append(yaw)


pitch_g= np.array(pitch_values)
yaw_g = np.array(yaw_values)

r_periapsis = a * (1.0 - e)
r_apoapsis  = a * (1.0 + e)

h_periapsis = r_periapsis - EARTH_R
h_apoapsis  = r_apoapsis  - EARTH_R



In [ ]:
plot(
    [t_sol, t_sol, time, time], [pitch_g,yaw_g, control.T[0], control.T[1]],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Flight Path Angle, Pitch_RSW",
    )

In [ ]:
data= client.get_channels(["h_periapsis", "h_apoapsis"])
plot(
    [t_sol,t_sol, time,time], [h_periapsis, h_apoapsis, data.T[0], data.T[1]],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="perigee and apogee altitudes",
    )

In [ ]:
e2= client.get_channels(["e"])

plot(
    [t_sol, time], [e, e2],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Eccentricity",
    )

In [ ]:
m2 = client.get_channels(["m"])
plot(
    [t_sol, time], [mass, m2],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Mass",
    )

In [ ]:
plot(
    [t_sol], [v_eci_mag],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="vel mag",
    )